# 20｜不用 `nn.MultiheadAttention/nn.Transformer`：手写 Seq2Seq Transformer

本笔记从 token/位置嵌入、缩放点积注意力、多头拆并、padding/causal mask、编码器/解码器块一路实现到 `Seq2SeqTransformer.forward` 与贪心解码。目标不是复刻大型语言模型，而是把“未来信息隔离、teacher forcing、权重共享与制品合同”变成可执行测试。

## 1. 张量与特殊 token 合同

- token 张量：`[B,T]`，`PAD=0, BOS=1, EOS=2`，普通 token 从 3 开始。
- 隐状态：`[B,T,D]`；多头后：`[B,H,T,D/H]`，要求 `D % H == 0`。
- 注意力允许掩码是 bool，形状可广播到 `[B,H,T_q,T_k]`，`True` 表示允许读取。
- 训练输入 `tgt_in` 以 BOS 开头；监督 `tgt_out` 相对左移一位，以 EOS 结束。
- 本实验只在固定小集合上受控过拟合，不把结果解释为序列泛化。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260729  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
PAD, BOS, EOS = 0, 1, 2  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert len({PAD, BOS, EOS}) == 3  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. 缩放点积注意力

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right)V.$$

对禁止位置，$M=-\infty$。工程上还必须处理“一整行都被屏蔽”：本合同保证每个 decoder 查询至少能看到 BOS，每个有效 encoder 查询至少能看到一个有效源 token。函数仍显式检查全屏蔽行，避免 softmax 产生 NaN。

In [ ]:
def scaled_dot_product_attention(q, k, v, allowed_mask=None):  # 定义本节可复用的核心函数。
    if q.shape[:-2] != k.shape[:-2] or k.shape != v.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("q/k/v 的 batch、head、通道合同不成立")  # 遇到非法合同立即显式失败。
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])  # 计算并保存当前步骤的中间状态。
    if allowed_mask is not None:  # 按当前条件选择后续控制路径。
        if allowed_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise TypeError("allowed_mask 必须是 bool")  # 遇到非法合同立即显式失败。
        expanded = torch.broadcast_to(allowed_mask, scores.shape)  # 计算并保存当前步骤的中间状态。
        if bool((~expanded.any(dim=-1)).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("存在完全不可见的 query")  # 遇到非法合同立即显式失败。
        scores = scores.masked_fill(~expanded, torch.finfo(scores.dtype).min)  # 计算并保存当前步骤的中间状态。
    weights = torch.softmax(scores, dim=-1)  # 计算并保存当前步骤的中间状态。
    return weights @ v, weights  # 返回当前分支计算出的结果。

q0 = torch.tensor([[[[1.0, 0.0]]]])  # 计算并保存当前步骤的中间状态。
k0 = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])  # 计算并保存当前步骤的中间状态。
v0 = torch.tensor([[[[3.0, 0.0], [0.0, 5.0]]]])  # 计算并保存当前步骤的中间状态。
ctx0, w0 = scaled_dot_product_attention(q0, k0, v0, torch.tensor([[[[True, False]]]]))  # 计算并保存当前步骤的中间状态。
assert ctx0.shape == (1, 1, 1, 2)  # 用受控断言验证关键不变量。
assert torch.allclose(ctx0, v0[:, :, :1])  # 用受控断言验证关键不变量。
assert torch.allclose(w0.sum(-1), torch.ones_like(w0.sum(-1)))  # 用受控断言验证关键不变量。

## 3. 多头拆分与合并

四个线性层分别产生 Q/K/V 与输出投影。`split_heads` 把 `[B,T,D]` 变为 `[B,H,T,d_h]`；`merge_heads` 做逆变换。我们没有调用任何预制注意力或 Transformer 层，因此每一步 shape 都可观察。

In [ ]:
class MultiHeadAttentionFromScratch(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, num_heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if d_model % num_heads:  # 按当前条件选择后续控制路径。
            raise ValueError("d_model 必须能被 num_heads 整除")  # 遇到非法合同立即显式失败。
        self.d_model, self.num_heads = d_model, num_heads  # 计算并保存当前步骤的中间状态。
        self.head_dim = d_model // num_heads  # 计算并保存当前步骤的中间状态。
        self.q_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.k_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.v_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.o_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。

    def split_heads(self, x):  # 定义本节可复用的核心函数。
        B, T, _ = x.shape  # 计算并保存当前步骤的中间状态。
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # 返回当前分支计算出的结果。

    def merge_heads(self, x):  # 定义本节可复用的核心函数。
        B, H, T, Dh = x.shape  # 计算并保存当前步骤的中间状态。
        if H != self.num_heads or Dh != self.head_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("head shape 不符合配置")  # 遇到非法合同立即显式失败。
        return x.transpose(1, 2).contiguous().view(B, T, self.d_model)  # 返回当前分支计算出的结果。

    def forward(self, query, key, value, allowed_mask=None, need_weights=False):  # 定义本节可复用的核心函数。
        q, k, v = map(self.split_heads, (self.q_proj(query), self.k_proj(key), self.v_proj(value)))  # 计算并保存当前步骤的中间状态。
        context, weights = scaled_dot_product_attention(q, k, v, allowed_mask)  # 计算并保存当前步骤的中间状态。
        out = self.o_proj(self.merge_heads(context))  # 计算并保存当前步骤的中间状态。
        return (out, weights) if need_weights else out  # 返回当前分支计算出的结果。

mha0 = MultiHeadAttentionFromScratch(12, 3)  # 计算并保存当前步骤的中间状态。
x0 = torch.randn(2, 5, 12)  # 计算并保存当前步骤的中间状态。
out0, weights0 = mha0(x0, x0, x0, torch.ones(2, 1, 5, 5, dtype=torch.bool), True)  # 计算并保存当前步骤的中间状态。
assert out0.shape == (2, 5, 12)  # 用受控断言验证关键不变量。
assert weights0.shape == (2, 3, 5, 5)  # 用受控断言验证关键不变量。
assert torch.allclose(weights0.sum(-1), torch.ones(2, 3, 5), atol=1e-6)  # 用受控断言验证关键不变量。

## 4. padding mask 与 causal mask 的组合

Encoder self-attention 屏蔽 padding key；decoder self-attention 同时要求 `key_position <= query_position` 且 key 非 padding；cross-attention 只屏蔽源 padding key。查询位置的 padding 最终会在 block 外清零，以免残差携带垃圾值。

注意：causal mask 的方向非常容易写反，必须用具体矩阵和“改未来不影响过去”的属性测试。

In [ ]:
def make_masks(src_tokens, tgt_tokens):  # 定义本节可复用的核心函数。
    if src_tokens.ndim != 2 or tgt_tokens.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("token 张量必须为 [B,T]")  # 遇到非法合同立即显式失败。
    B, S = src_tokens.shape  # 计算并保存当前步骤的中间状态。
    _, T = tgt_tokens.shape  # 计算并保存当前步骤的中间状态。
    src_key = (src_tokens != PAD)[:, None, None, :]           # [B,1,1,S]；中文说明：该行遵循既定约束。
    tgt_key = (tgt_tokens != PAD)[:, None, None, :]           # [B,1,1,T]；中文说明：该行遵循既定约束。
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=tgt_tokens.device))[None, None]  # 计算并保存当前步骤的中间状态。
    tgt_self = tgt_key & causal  # 计算并保存当前步骤的中间状态。
    cross = src_key  # 计算并保存当前步骤的中间状态。
    return src_key, tgt_self, cross  # 返回当前分支计算出的结果。

src_probe = torch.tensor([[4, 5, PAD], [6, 7, 8]])  # 计算并保存当前步骤的中间状态。
tgt_probe = torch.tensor([[BOS, 9, PAD], [BOS, 8, 7]])  # 计算并保存当前步骤的中间状态。
src_mask0, tgt_mask0, cross_mask0 = make_masks(src_probe, tgt_probe)  # 计算并保存当前步骤的中间状态。
assert src_mask0.shape == (2, 1, 1, 3)  # 用受控断言验证关键不变量。
assert tgt_mask0.shape == (2, 1, 3, 3)  # 用受控断言验证关键不变量。
assert not tgt_mask0[0, 0, 0, 1]  # 用受控断言验证关键不变量。
assert tgt_mask0[0, 0, 1, 0]  # 用受控断言验证关键不变量。
assert not tgt_mask0[0, 0, 2, 2]  # 用受控断言验证关键不变量。
assert torch.equal(src_mask0, cross_mask0)  # 用受控断言验证关键不变量。

## 5. Token + position embedding

可学习位置表把顺序注入模型。token embedding 乘 $\sqrt{D}$，避免其初始尺度相对位置向量过小。超过 `max_len` 直接拒绝，不能静默截断，因为截断会改变监督对齐。

In [ ]:
class TokenPositionEmbedding(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, d_model, max_len):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.token = nn.Embedding(vocab_size, d_model, padding_idx=PAD)  # 计算并保存当前步骤的中间状态。
        self.position = nn.Embedding(max_len, d_model)  # 计算并保存当前步骤的中间状态。
        self.d_model, self.max_len = d_model, max_len  # 计算并保存当前步骤的中间状态。

    def forward(self, tokens):  # 定义本节可复用的核心函数。
        if tokens.shape[1] > self.max_len:  # 按当前条件选择后续控制路径。
            raise ValueError("序列超过位置表长度")  # 遇到非法合同立即显式失败。
        pos = torch.arange(tokens.shape[1], device=tokens.device)  # 计算并保存当前步骤的中间状态。
        return self.token(tokens) * math.sqrt(self.d_model) + self.position(pos)[None]  # 返回当前分支计算出的结果。

class FeedForward(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, d_ff):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))  # 计算并保存当前步骤的中间状态。
    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.net(x)  # 返回当前分支计算出的结果。

class EncoderBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, heads, d_ff):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.attn = MultiHeadAttentionFromScratch(d_model, heads)  # 计算并保存当前步骤的中间状态。
        self.ffn = FeedForward(d_model, d_ff)  # 计算并保存当前步骤的中间状态。
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。
    def forward(self, x, src_mask, query_valid):  # 定义本节可复用的核心函数。
        x = self.norm1(x + self.attn(x, x, x, src_mask))  # 计算并保存当前步骤的中间状态。
        x = self.norm2(x + self.ffn(x))  # 计算并保存当前步骤的中间状态。
        return x * query_valid.unsqueeze(-1)  # 返回当前分支计算出的结果。

class DecoderBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, heads, d_ff):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.self_attn = MultiHeadAttentionFromScratch(d_model, heads)  # 计算并保存当前步骤的中间状态。
        self.cross_attn = MultiHeadAttentionFromScratch(d_model, heads)  # 计算并保存当前步骤的中间状态。
        self.ffn = FeedForward(d_model, d_ff)  # 计算并保存当前步骤的中间状态。
        self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(d_model) for _ in range(3))  # 计算并保存当前步骤的中间状态。
    def forward(self, x, memory, self_mask, cross_mask, query_valid):  # 定义本节可复用的核心函数。
        x = self.norm1(x + self.self_attn(x, x, x, self_mask))  # 计算并保存当前步骤的中间状态。
        x = self.norm2(x + self.cross_attn(x, memory, memory, cross_mask))  # 计算并保存当前步骤的中间状态。
        x = self.norm3(x + self.ffn(x))  # 计算并保存当前步骤的中间状态。
        return x * query_valid.unsqueeze(-1)  # 返回当前分支计算出的结果。

emb0 = TokenPositionEmbedding(20, 12, 8)  # 计算并保存当前步骤的中间状态。
assert emb0(torch.tensor([[BOS, 4, EOS]])).shape == (1, 3, 12)  # 用受控断言验证关键不变量。
assert emb0.token.padding_idx == PAD  # 用受控断言验证关键不变量。

## 6. Encoder、Decoder 与权重共享

模型先编码源序列，再以 causal self-attention 和 cross-attention 解码。输出投影与目标 token embedding 共享同一个 `Parameter`：这减少参数并要求词表与隐藏维度兼容。共享不能只靠数值相等验证，要检查对象/存储地址。

单个带 bias 的多头注意力参数量为 $4(D^2+D)$；两层 FFN 为 $2DD_{ff}+D_{ff}+D$。总参数还包括两个词嵌入/位置表、LayerNorm 与层数倍数。代码用 `model.parameters()` 对唯一 Parameter 求和，因此被绑定的输出矩阵不会重复计数。

In [ ]:
class Seq2SeqTransformer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, d_model=32, heads=4, d_ff=64, layers=1, max_len=16):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.config = dict(vocab_size=vocab_size, d_model=d_model, heads=heads,  # 计算并保存当前步骤的中间状态。
                           d_ff=d_ff, layers=layers, max_len=max_len)  # 计算并保存当前步骤的中间状态。
        self.src_embed = TokenPositionEmbedding(vocab_size, d_model, max_len)  # 计算并保存当前步骤的中间状态。
        self.tgt_embed = TokenPositionEmbedding(vocab_size, d_model, max_len)  # 计算并保存当前步骤的中间状态。
        self.encoders = nn.ModuleList([EncoderBlock(d_model, heads, d_ff) for _ in range(layers)])  # 计算并保存当前步骤的中间状态。
        self.decoders = nn.ModuleList([DecoderBlock(d_model, heads, d_ff) for _ in range(layers)])  # 计算并保存当前步骤的中间状态。
        self.output = nn.Linear(d_model, vocab_size, bias=False)  # 计算并保存当前步骤的中间状态。
        self.output.weight = self.tgt_embed.token.weight  # 计算并保存当前步骤的中间状态。

    def encode(self, src):  # 定义本节可复用的核心函数。
        src_mask, _, _ = make_masks(src, torch.full((src.shape[0], 1), BOS, device=src.device))  # 计算并保存当前步骤的中间状态。
        valid = src != PAD  # 计算并保存当前步骤的中间状态。
        memory = self.src_embed(src) * valid.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        for block in self.encoders:  # 遍历输入元素以累积或检查结果。
            memory = block(memory, src_mask, valid)  # 计算并保存当前步骤的中间状态。
        return memory, src_mask  # 返回当前分支计算出的结果。

    def decode(self, tgt_in, memory, src_mask):  # 定义本节可复用的核心函数。
        _, self_mask, cross = make_masks(torch.ones(src_mask.shape[0], src_mask.shape[-1],  # 计算并保存当前步骤的中间状态。
                                                    dtype=torch.long, device=tgt_in.device), tgt_in)  # 计算并保存当前步骤的中间状态。
        cross = src_mask  # 计算并保存当前步骤的中间状态。
        valid = tgt_in != PAD  # 计算并保存当前步骤的中间状态。
        x = self.tgt_embed(tgt_in) * valid.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        for block in self.decoders:  # 遍历输入元素以累积或检查结果。
            x = block(x, memory, self_mask, cross, valid)  # 计算并保存当前步骤的中间状态。
        return self.output(x)  # 返回当前分支计算出的结果。

    def forward(self, src, tgt_in):  # 定义本节可复用的核心函数。
        memory, src_mask = self.encode(src)  # 计算并保存当前步骤的中间状态。
        return self.decode(tgt_in, memory, src_mask)  # 返回当前分支计算出的结果。

model20 = Seq2SeqTransformer(vocab_size=14)  # 计算并保存当前步骤的中间状态。
dummy_logits = model20(torch.tensor([[4, 5, EOS]]), torch.tensor([[BOS, 5, 4]]))  # 计算并保存当前步骤的中间状态。
parameter_count20 = sum(p.numel() for p in model20.parameters())  # 共享 Parameter 只计一次
assert dummy_logits.shape == (1, 3, 14)  # 用受控断言验证关键不变量。
assert parameter_count20 > 10_000  # 用受控断言验证关键不变量。
assert model20.output.weight is model20.tgt_embed.token.weight  # 用受控断言验证关键不变量。
assert model20.output.weight.data_ptr() == model20.tgt_embed.token.weight.data_ptr()  # 用受控断言验证关键不变量。
assert not any(isinstance(m, (nn.MultiheadAttention, nn.Transformer)) for m in model20.modules())  # 用受控断言验证关键不变量。
print({"unique_trainable_parameters": parameter_count20})  # 执行当前语句以推进本节示例。

## 7. Reverse toy：shifted target 与 teacher forcing

源序列是 3 个普通 token 加 EOS；目标是把 3 个 token 逆序后加 EOS。训练输入在最前放 BOS，并移除最后一个监督 token。这个数据集故意很小，目的只是检测 forward、mask、loss 和 decode 是否闭环。

`ignore_index=PAD` 让填充位置不进入损失；源/目标仍保留 EOS，防止解码只能依赖固定长度。

In [ ]:
def build_reverse_data():  # 定义本节可复用的核心函数。
    sequences = [  # 计算并保存当前步骤的中间状态。
        [3, 4, 5], [3, 5, 6], [4, 6, 7], [5, 7, 8],  # 执行当前语句以推进本节示例。
        [6, 8, 9], [7, 9, 10], [8, 10, 11], [9, 11, 12],  # 执行当前语句以推进本节示例。
        [3, 7, 11], [4, 8, 12], [5, 9, 3], [6, 10, 4],  # 执行当前语句以推进本节示例。
    ]  # 执行当前语句以推进本节示例。
    src = torch.tensor([s + [EOS] for s in sequences])  # 计算并保存当前步骤的中间状态。
    target = torch.tensor([list(reversed(s)) + [EOS] for s in sequences])  # 计算并保存当前步骤的中间状态。
    tgt_in = torch.cat([torch.full((len(sequences), 1), BOS), target[:, :-1]], dim=1)  # 计算并保存当前步骤的中间状态。
    return src, tgt_in, target  # 返回当前分支计算出的结果。

src20, tgt_in20, tgt_out20 = build_reverse_data()  # 计算并保存当前步骤的中间状态。
assert src20.shape == tgt_in20.shape == tgt_out20.shape == (12, 4)  # 用受控断言验证关键不变量。
assert torch.equal(tgt_in20[:, 0], torch.full((12,), BOS))  # 用受控断言验证关键不变量。
assert torch.equal(tgt_out20[:, -1], torch.full((12,), EOS))  # 用受控断言验证关键不变量。
assert torch.equal(tgt_out20[0, :3], src20[0, :3].flip(0))  # 用受控断言验证关键不变量。

## 8. 属性测试：修改未来 token 不得改变过去 logits

在 `eval` 模式下构造两个目标前缀：位置 0、1 相同，从位置 2 开始不同。若 causal mask 正确，则前两个位置的 logits 应完全一致；若误用双向 self-attention，这一断言会失败。

In [ ]:
model20.eval()  # 执行当前语句以推进本节示例。
prefix_a = torch.tensor([[BOS, 5, 6, 7]])  # 计算并保存当前步骤的中间状态。
prefix_b = torch.tensor([[BOS, 5, 12, 11]])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    future_a = model20(src20[:1], prefix_a)  # 计算并保存当前步骤的中间状态。
    future_b = model20(src20[:1], prefix_b)  # 计算并保存当前步骤的中间状态。

assert torch.allclose(future_a[:, :2], future_b[:, :2], atol=1e-6)  # 用受控断言验证关键不变量。
assert not torch.allclose(future_a[:, 2:], future_b[:, 2:], atol=1e-5)  # 用受控断言验证关键不变量。
assert torch.isfinite(future_a).all()  # 用受控断言验证关键不变量。

## 9. 受控过拟合训练

使用全批量 AdamW 与 token 交叉熵。每次反向传播后检查梯度有限，并裁剪全局范数。验收看 token 准确率、精确序列匹配率和 loss 降幅；这些数值仅属于训练集合。

In [ ]:
torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
model20 = Seq2SeqTransformer(vocab_size=14, d_model=32, heads=4, d_ff=64, layers=1, max_len=8)  # 计算并保存当前步骤的中间状态。
opt20 = torch.optim.AdamW(model20.parameters(), lr=0.015, weight_decay=0.0)  # 计算并保存当前步骤的中间状态。
losses20 = []  # 计算并保存当前步骤的中间状态。
model20.train()  # 执行当前语句以推进本节示例。
for step in range(320):  # 遍历输入元素以累积或检查结果。
    opt20.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits = model20(src20, tgt_in20)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt_out20.reshape(-1), ignore_index=PAD)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        grad_tensors = [p.grad for p in model20.parameters() if p.grad is not None]  # 计算并保存当前步骤的中间状态。
        assert grad_tensors and all(torch.isfinite(g).all() for g in grad_tensors)  # 用受控断言验证关键不变量。
        assert sum(float(g.abs().sum()) for g in grad_tensors) > 0  # 用受控断言验证关键不变量。
    torch.nn.utils.clip_grad_norm_(model20.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    opt20.step()  # 执行当前语句以推进本节示例。
    losses20.append(float(loss.detach()))  # 执行当前语句以推进本节示例。

model20.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    trained_logits = model20(src20, tgt_in20)  # 计算并保存当前步骤的中间状态。
    teacher_pred = trained_logits.argmax(-1)  # 计算并保存当前步骤的中间状态。
token_acc20 = (teacher_pred == tgt_out20).float().mean().item()  # 计算并保存当前步骤的中间状态。
exact_teacher20 = (teacher_pred == tgt_out20).all(1).float().mean().item()  # 计算并保存当前步骤的中间状态。

assert losses20[-1] < losses20[0] * 0.15  # 用受控断言验证关键不变量。
assert token_acc20 >= 0.98  # 用受控断言验证关键不变量。
assert exact_teacher20 >= 0.90  # 用受控断言验证关键不变量。
assert torch.isfinite(trained_logits).all()  # 用受控断言验证关键不变量。
print({"loss_first": losses20[0], "loss_last": losses20[-1],  # 执行当前语句以推进本节示例。
       "teacher_token_acc": token_acc20, "teacher_exact": exact_teacher20})  # 执行当前语句以推进本节示例。

## 10. 贪心解码不是 teacher forcing

推理时没有真实前一 token。每步重新运行 decoder，取最后位置 argmax，并把已生成序列拼回输入；所有样本遇到 EOS 后结束。真实服务还需要最大长度、非法 token 过滤、beam cache 和批处理调度。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def greedy_decode(model, src, max_new_tokens):  # 定义本节可复用的核心函数。
    model.eval()  # 执行当前语句以推进本节示例。
    memory, src_mask = model.encode(src)  # 计算并保存当前步骤的中间状态。
    generated = torch.full((src.shape[0], 1), BOS, dtype=torch.long, device=src.device)  # 计算并保存当前步骤的中间状态。
    finished = torch.zeros(src.shape[0], dtype=torch.bool, device=src.device)  # 计算并保存当前步骤的中间状态。
    for _ in range(max_new_tokens):  # 遍历输入元素以累积或检查结果。
        next_token = model.decode(generated, memory, src_mask)[:, -1].argmax(-1)  # 计算并保存当前步骤的中间状态。
        next_token = torch.where(finished, torch.full_like(next_token, EOS), next_token)  # 计算并保存当前步骤的中间状态。
        generated = torch.cat([generated, next_token[:, None]], dim=1)  # 计算并保存当前步骤的中间状态。
        finished |= next_token.eq(EOS)  # 计算并保存当前步骤的中间状态。
        if bool(finished.all()):  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
    return generated[:, 1:]  # 返回当前分支计算出的结果。

decoded20 = greedy_decode(model20, src20, max_new_tokens=4)  # 计算并保存当前步骤的中间状态。
greedy_exact20 = (decoded20 == tgt_out20).all(1).float().mean().item()  # 计算并保存当前步骤的中间状态。
assert decoded20.shape == tgt_out20.shape  # 用受控断言验证关键不变量。
assert greedy_exact20 >= 0.90  # 用受控断言验证关键不变量。
assert torch.equal(decoded20[:, -1], torch.full((12,), EOS))  # 用受控断言验证关键不变量。
print({"greedy_exact_on_controlled_set": greedy_exact20, "example": decoded20[0].tolist()})  # 执行当前语句以推进本节示例。

## 11. 常见失败模式

- 只给 loss 加 `ignore_index`，却不在注意力中屏蔽 PAD：padding 仍可污染其他 token。
- causal 三角矩阵方向写反：训练指标异常好，但自回归解码崩溃。
- 把 `tgt_out` 原样喂给 decoder：当前位置直接看到答案，属于标签泄漏。
- 共享权重时先创建 optimizer、后替换 Parameter：optimizer 可能仍追踪旧权重。
- 只报告 teacher-forcing token accuracy：它不能替代真实 greedy/beam 序列指标。
- 无最大生成长度或 EOS 策略：坏模型会无限生成。

## 12. 制品与版本合同

manifest 必须包含特殊 token、最大长度、位置编码类型、归一化顺序、层数/头数、权重共享方式、词表哈希和 PyTorch 版本。仅有 `state_dict` 无法判断同一组矩阵应按哪种 mask 语义运行。

In [ ]:
manifest20 = {  # 计算并保存当前步骤的中间状态。
    "artifact": "seq2seq_transformer_from_scratch",  # 执行当前语句以推进本节示例。
    "schema_version": 1,  # 执行当前语句以推进本节示例。
    "config": model20.config,  # 执行当前语句以推进本节示例。
    "special_tokens": {"pad": PAD, "bos": BOS, "eos": EOS},  # 执行当前语句以推进本节示例。
    "attention_mask": "bool_true_means_allowed",  # 执行当前语句以推进本节示例。
    "norm": "post_norm",  # 执行当前语句以推进本节示例。
    "target_contract": "BOS + target[:-1]",  # 执行当前语句以推进本节示例。
    "weight_tying": "output.weight is tgt_embed.token.weight",  # 执行当前语句以推进本节示例。
    "torch_version": torch.__version__,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
buf20 = io.BytesIO()  # 计算并保存当前步骤的中间状态。
torch.save({"manifest": manifest20, "state_dict": model20.state_dict()}, buf20)  # 执行当前语句以推进本节示例。
artifact20 = buf20.getvalue()  # 计算并保存当前步骤的中间状态。
sha20 = hashlib.sha256(artifact20).hexdigest()  # 计算并保存当前步骤的中间状态。
buf20.seek(0)  # 执行当前语句以推进本节示例。
loaded20 = torch.load(buf20, map_location="cpu", weights_only=False)  # 计算并保存当前步骤的中间状态。
clone20 = Seq2SeqTransformer(**loaded20["manifest"]["config"])  # 计算并保存当前步骤的中间状态。
clone20.load_state_dict(loaded20["state_dict"])  # 执行当前语句以推进本节示例。
clone20.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    clone_logits20 = clone20(src20, tgt_in20)  # 计算并保存当前步骤的中间状态。

assert len(sha20) == 64  # 用受控断言验证关键不变量。
assert loaded20["manifest"]["special_tokens"]["bos"] == BOS  # 用受控断言验证关键不变量。
assert clone20.output.weight is clone20.tgt_embed.token.weight  # 用受控断言验证关键不变量。
assert torch.allclose(clone_logits20, trained_logits, atol=1e-7)  # 用受控断言验证关键不变量。
print({"sha256": sha20[:16] + "…", "bytes": len(artifact20)})  # 执行当前语句以推进本节示例。

## 13. 生产替换与安全

教学实现用 Python 组合算子，便于理解，不保证 FlashAttention、KV cache、混合精度、分布式训练或 ONNX 导出性能。生产替换必须保留这里的 mask 属性测试与 decode 回归样例。

输入侧应限制 token 数、batch、生成长度与词表范围；按租户隔离 KV cache；记录截断率、EOS 到达率、生成长度、延迟分位数、无穷/NaN 计数和模型指纹。不要反序列化不可信 checkpoint。

## 14. 原始资料

- Vaswani et al., *Attention Is All You Need* (2017)：https://arxiv.org/abs/1706.03762
- PyTorch `nn.Module`：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch broadcasting semantics：https://pytorch.org/docs/stable/notes/broadcasting.html
- PyTorch `CrossEntropyLoss`：https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html

架构和缩放点积公式来自原论文；这里的类、mask 约定和测试为独立教学实现。

In [ ]:
# 最终合同：非法 head、非法 mask、超长位置都必须失败。
def raises_value_error(fn):  # 定义本节可复用的核心函数。
    try:  # 尝试执行可能失败的受控操作。
        fn()  # 执行当前语句以推进本节示例。
        return False  # 返回当前分支计算出的结果。
    except (ValueError, TypeError):  # 捕获预期异常并验证失败分支。
        return True  # 返回当前分支计算出的结果。

assert raises_value_error(lambda: MultiHeadAttentionFromScratch(10, 3))  # 用受控断言验证关键不变量。
assert raises_value_error(lambda: scaled_dot_product_attention(q0, k0, v0,  # 用受控断言验证关键不变量。
                              torch.zeros(1, 1, 1, 2, dtype=torch.bool)))  # 计算并保存当前步骤的中间状态。
assert raises_value_error(lambda: emb0(torch.ones(1, 9, dtype=torch.long)))  # 用受控断言验证关键不变量。
assert model20.config["d_model"] % model20.config["heads"] == 0  # 用受控断言验证关键不变量。
assert manifest20["attention_mask"] == "bool_true_means_allowed"  # 用受控断言验证关键不变量。
assert losses20[-1] < 0.1  # 用受控断言验证关键不变量。
assert greedy_exact20 <= 1.0  # 用受控断言验证关键不变量。
assert sha20 == hashlib.sha256(artifact20).hexdigest()  # 用受控断言验证关键不变量。
assert all(torch.isfinite(p).all() for p in model20.parameters())  # 用受控断言验证关键不变量。
print("Notebook 20：全部合同测试通过。")  # 执行当前语句以推进本节示例。